# Day 1-6: Tools You Don't Own

Ludwig's internet went out at 9:40 PM. The Summit Wireless support bot told him there were **no known outages**, so he restarted his router three times and booked a $79 technician visit. There was an outage. It had been reported 25 minutes earlier.

In this notebook you play **both** teams:

- **Network Ops**, who own the outage data and publish it as an MCP server
- **The chatbot team** (you), who connect an agent to it without ever seeing their code

Work with your partner and stop after each section until you both have it working.

## 1. Add your API key

Same as Day 1: in the **Secrets** panel (the key icon on the left), make sure `GOOGLE_API_KEY` exists and **Notebook access** is on. Then run the cell.

In [ ]:
import os
from google.colab import userdata

os.environ["GOOGLE_API_KEY"] = userdata.get("GOOGLE_API_KEY")
print("API key loaded.")

## 2. Install

The `mcp` extra adds `MCPAdapter` to LangChain and pulls in **FastMCP**, the library for writing MCP servers.

In [ ]:
%pip install -q -U "langchain[mcp]>=1.4" langchain-google-genai

## 3. Ludwig's bug, in 15 lines

The top half is Network Ops' API **after** last week's change. Pretend you can't see it: it's in their repo, written in Java, on their release schedule.

The bottom half is the chatbot team's wrapper, written months ago from Network Ops' docs.

In [ ]:
# ---- Network Ops' API: their code, their repo ----
OUTAGE_DB = [{"postal_code": "83440", "summary": "Fiber cut reported at 9:15 PM. Crews on site, ETA 11:30 PM."}]

def network_ops_api(params: dict) -> list:
    postal_code = params.get("postal_code")    # last week this was params.get("zip")
    return [o for o in OUTAGE_DB if o["postal_code"] == postal_code]

# ---- The chatbot team's wrapper: your code, written in March ----
def check_outage_wrapper(zip_code: str) -> str:
    """Check whether there is a known network outage in a ZIP code."""
    outages = network_ops_api({"zip": zip_code})
    if not outages:
        return "No known outages."
    return outages[0]["summary"]

check_outage_wrapper("83440")

No error. No crash. Just a confident, wrong answer, and a model would believe it, because it came from a tool.

**Discuss:** How long could this bug go unnoticed? Who would notice first?

## 4. Be Network Ops: publish an MCP server

Now Network Ops publishes the outage lookup themselves, as an MCP server. This is **their** file, not yours.

`%%writefile` saves the cell to a file on the Colab machine instead of running it. This is the version from **before** the rename, so the parameter is still `zip_code`.

In [ ]:
%%writefile outage_server.py
from fastmcp import FastMCP

mcp = FastMCP("Network Ops")

OUTAGES = {"83440": "Fiber cut reported at 9:15 PM. Crews on site, ETA 11:30 PM."}

@mcp.tool
def check_outage(zip_code: str) -> str:
    """Check whether there is a known network outage in a ZIP code."""
    return OUTAGES.get(zip_code, "No known outages.")

if __name__ == "__main__":
    mcp.run()

## 5. Be the chatbot team: connect and look

Back to being you. Connect to Network Ops' server and ask what tools it has. **No LLM yet.**

- `Path("outage_server.py")` means "launch this script and talk to it" (stdio). You never import it.
- Colab lets you use `await` right in a cell. In a regular `.py` script you'd need `asyncio.run(...)`.
- The FastMCP banner and the "`langchain.mcp` is in beta" warning are normal.

In [ ]:
from pathlib import Path
from langchain.mcp import MCPAdapter

async with MCPAdapter(Path("outage_server.py")) as adapter:
    tools = await adapter.list_tools()

for t in tools:
    print(t.name, t.args)
    print("   ", t.description)

Where did the name, arguments, and description come from? You didn't type any of them.

## 6. Call the tool directly

Before you involve a model, check the plumbing works. MCP tools are **async-only**, so it's `ainvoke`, not `invoke`.

In [ ]:
await tools[0].ainvoke({"zip_code": "83440"})

## 7. Give it to an agent

Ask Ludwig's question.

In [ ]:
from langchain.agents import create_agent

SYSTEM_PROMPT = (
    "You are the customer support agent for Summit Wireless, an internet provider. "
    "Before suggesting troubleshooting or a technician visit, check for a known outage."
)
LUDWIG = "My internet in 83440 has been out for 20 minutes. What's going on?"

agent = create_agent("google_genai:gemini-flash-lite-latest", tools, system_prompt=SYSTEM_PROMPT)

result = await agent.ainvoke({"messages": [{"role": "user", "content": LUDWIG}]})
print(result["messages"][-1].text)

**Try it:** change `ainvoke` to `invoke` (and remove the `await`), run it, and read the error. Then change it back.

## 8. Be Network Ops again: ship the rename

Network Ops renames `zip_code` to `postal_code`: the exact change that broke Ludwig's bot. This time they make it **in their own server**.

In [ ]:
%%writefile outage_server.py
from fastmcp import FastMCP

mcp = FastMCP("Network Ops")

OUTAGES = {"83440": "Fiber cut reported at 9:15 PM. Crews on site, ETA 11:30 PM."}

@mcp.tool
def check_outage(postal_code: str) -> str:
    """Check whether there is a known network outage in a postal code."""
    return OUTAGES.get(postal_code, "No known outages.")

if __name__ == "__main__":
    mcp.run()

Now be the chatbot team. This cell is **the same code** as sections 5 and 7, with nothing changed. Run it.

In [ ]:
async with MCPAdapter(Path("outage_server.py")) as adapter:
    tools = await adapter.list_tools()
print([(t.name, t.args) for t in tools])

agent = create_agent("google_genai:gemini-flash-lite-latest", tools, system_prompt=SYSTEM_PROMPT)
result = await agent.ainvoke({"messages": [{"role": "user", "content": LUDWIG}]})
print(result["messages"][-1].text)

**Discuss:**

- How many lines of *your* code changed?
- Why didn't the rename break anything this time? (Hint: when does your agent learn the argument names?)
- What would have happened to Ludwig?

## 9. More than one server

Real agents connect to several servers. Add one you definitely don't own: the **LangChain docs** server, which LangChain runs on the internet. You connect with a URL instead of a file path.

With several servers, each tool's name gets its server's name as a prefix, so tools from different servers can't collide.

In [ ]:
config = {"mcpServers": {
    "network": {"command": "python", "args": ["outage_server.py"]},   # launched here (stdio)
    "docs":    {"url": "https://docs.langchain.com/mcp"},             # already running (HTTP)
}}

async with MCPAdapter(config) as adapter:
    tools = await adapter.list_tools()

for t in tools:
    print(t.name)

In [ ]:
agent = create_agent("google_genai:gemini-flash-lite-latest", tools, system_prompt=SYSTEM_PROMPT)

question = "Is there an outage in 83440? Also, I'm building a support bot: how do I add short-term memory to a LangChain agent? Keep it brief."
result = await agent.ainvoke({"messages": [{"role": "user", "content": question}]})

for m in result["messages"]:
    for call in getattr(m, "tool_calls", []):
        print("called:", call["name"])
print()
print(result["messages"][-1].text)

## 10. Keep going

Pick at least one:

1. **Be Network Ops:** add a second `@mcp.tool` to `outage_server.py` (for example `list_outages()` or `estimate_restore_time(postal_code)`). Rewrite the file and rerun section 9. What did you change on the agent side?
2. **Break it on purpose:** change the tool's docstring to something vague, like `"""Does network stuff."""`. Does the agent still call it for Ludwig's question? Who's responsible for good descriptions now?
3. **Find a server you'd actually use:** browse the servers from the prep reading. Which one would you add to your Project 1 agent, and is it stdio (a command) or HTTP (a URL)?